# OpenAI extraction workflow

This template builds a small corpus and extracts structured records with OpenAI text and vision profiles. Set `OPENAI_API_KEY` in the environment before starting Jupyter; never paste it into this notebook. Replace the model placeholder with a model available to your account.

In [ ]:
%env PM_DB=papers.db
%env PM_QUERY=lithium solid electrolyte
%env PM_RECIPE=sse
%env PM_MODEL=YOUR_OPENAI_MODEL
%env PM_OUTPUT=temp_openai_materials.csv
%env PM_FINAL=openai_materials.csv

## Configure model profiles

The same model may be used for both profiles only if it accepts images. Otherwise set separate text and vision identifiers.

In [ ]:
%%bash
set -euo pipefail
test "$PM_MODEL" != "YOUR_OPENAI_MODEL"
pm_model_config text --provider openai --model "$PM_MODEL"
pm_model_config vision --provider openai --model "$PM_MODEL"
pm_model_status

## Build and inspect the corpus

OpenAlex can search without a key. Configure other providers separately if you want broader coverage. Start with a small count, inspect the corpus, and scale only after the complete workflow succeeds.

In [ ]:
%%bash
set -euo pipefail
pm_search "$PM_QUERY" "$PM_DB" --source openalex --count 25
pm_download "$PM_DB" --format both
pm_corpus_stats "$PM_DB"

## Scrape and store

`text-images` uses downloaded text and PDF-derived images. Change the mode to `text` for a cheaper first run. The same recipe is passed to storage so aliases and unit conversions remain consistent.

In [ ]:
%%bash
set -euo pipefail
pm_scrape "$PM_DB" "$PM_RECIPE" --mode text-images --image-context paper-text --count 5 --output "$PM_OUTPUT"
pm_store "$PM_DB" "$PM_OUTPUT" "$PM_FINAL" "$PM_RECIPE" --assume-yes
pm_status "$PM_DB"